In [1]:
import os

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch

import T_perturb
from T_perturb.Perturb.datamodule import PerturberDataModule
from T_perturb.Perturb.trainer import PerturberTrainer

from T_perturb.src.utils import label_encoder
from T_perturb.tests.test_cellgen_training import dummy_dataset
from T_perturb.tests.test_countdecoder_training import dummy_cell_gene_matrix

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
seed_no = 42
pl.seed_everything(seed_no)
torch.manual_seed(seed_no)

Seed set to 42


In [3]:
if os.getcwd().split('/')[-1] != 'healthy_imm_expr':
    # set working directory to root of repository
    os.chdir('/lustre/scratch126/cellgen/team361/kl11/t_generative/')

In [4]:
tgt_vocab_size = 101  # +1 for padding token
num_samples = 100
num_genes = 100
max_seq_length = 50
n_total_tps = 2
num_samples = 100
batch_size = 4
pred_tps = [1, 2]
context_tps = [1, 2]
d_model = 12

genes_to_perturb = [70]
perturbation_token = 1


In [5]:
src_counts = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
)
src_dataset = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
)
tgt_counts_dict = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
    total_time_steps=n_total_tps,
)
tgt_datasets = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
    total_time_steps=n_total_tps,
)

In [ ]:
print(tgt_datasets['tgt_dataset_t1']['input_ids'][1])

[46, 30, 17, 15, 35, 79, 61, 95, 62, 27, 23, 69, 87, 65, 68, 100, 73, 33, 22, 9, 81, 56, 63, 97, 57, 77, 75, 72, 11, 5, 2, 10, 86, 85, 29, 94, 74, 90, 43, 49, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [7]:
# create dictionnary of metadata for classifier-free guidance
condition_dict = {}
conditions = ['cell_type']
token_no = 0
token_no += 1
condition_dict['timepoint'] = {
    f't_{i}': i+token_no for i in range(0, n_total_tps + 1)
}
token_no += n_total_tps+1
for condition in conditions:
    condition_dict[condition] = {
        cell_type: i+token_no for i, cell_type in enumerate(
            tgt_datasets['tgt_dataset_t2'].unique(condition)
        )
    }
    token_no += len(condition_dict[condition])
# add timepoint to metadata

In [8]:
# check if cuda is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [9]:
import importlib
import T_perturb.src.utils
importlib.reload(T_perturb.Perturb.trainer)
importlib.reload(T_perturb.Perturb.datamodule)
importlib.reload(T_perturb.src.utils)
from T_perturb.Perturb.trainer import PerturberTrainer
from T_perturb.Perturb.datamodule import PerturberDataModule

trainer_params = {
    'tgt_vocab_size': tgt_vocab_size,
    'd_model': d_model,
    'num_heads': 4,
    'num_layers': 1,
    'd_ff': 8,
    'max_seq_length': max_seq_length + 10,
    'dropout': 0.0,
    'pred_tps': pred_tps,
    'context_tps': context_tps,
    'n_total_tps': n_total_tps,
    'precision': 'high',
    'mask_scheduler': 'pow',
    'output_dir': './T_perturb/T_perturb/tests/res',
    'encoder': 'Transformer_encoder',
    'var_list': None,
    'context_mode': False,
    'temperature':1.5,
    'iterations': 19,
    'sequence_length': max_seq_length - 10,
    'pos_encoding_mode': 'time_pos_sin',
    'return_attn': False,
    'mask_scheduler': 'cosine',
    'condition_dict': condition_dict,
    'batch_size': batch_size,
    'condition': True,
    'context_mode': False
}
decoder_module = PerturberTrainer(
    **trainer_params
)
data_module = PerturberDataModule(
    batch_size=batch_size,
    src_counts=src_counts,
    tgt_counts_dict=tgt_counts_dict,
    src_dataset=src_dataset,
    tgt_datasets=tgt_datasets,
    num_workers=1,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    split=False,
    max_len=max_seq_length,
    var_list=['cell_type']
)
data_module.setup()

Using CPU for training
cls_token_1 tensor([101])
cls_token_2 tensor([102])
condition_dict {'timepoint': {'t_0': 1, 't_1': 2, 't_2': 3}, 'cell_type': {'B': 4, 'C': 5, 'A': 6}}
Start datamodule
Start perturbation ...
- Condition to perturb: None
- Condition observation key: None
- Filter mode: include



In [10]:
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
trainer = pl.Trainer(
    limit_test_batches=1,  # Limit to a single batch for quick testing
    logger=False,
    accelerator=accelerator,
    devices=1,  # inference only on one gpu
)
trainer.fit(
    decoder_module, 
    data_module,
    )


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.
/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/pytorch_lightning/loops/utilities.py:72: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /lustre/scratch126/cellgen/team361/kl11/t_generative/checkpoints exists and is not empty.

  | Name         | Type             | Params | Mode 
----------------------------------------------------------
0 | transformer  | PerturberMasking | 88.3 K | train
1 | masking_loss | CrossEntropyLoss | 0      | train
2 | perplexity   | Perplexity       | 0      | train
3 | mse          | MeanSquaredError | 0  

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/pytorch_lightning/utilities/data.py:105: Total length of `list` across ranks is zero. Please make sure this was your intention.
/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Epoch 0:   0%|          | 0/25 [00:00<?, ?it/s] 

/lustre/scratch126/cellgen/team361/kl11/t_generative/T_perturb/T_perturb/Perturb/trainer.py:144: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.condition_ids[:, j] = torch.tensor(


tgt_input_id tensor([[ 29,   5,  16,  65,  31,  50,  58,  99,  73,  62,  64,  85,  67,  26,
           6,  43,  12,  93,  44,  46,  82,  72,  30,  81,  22,  97,  69,  91,
          80,  24,  45,   9,  34,  27,  36,  74,  37,  40,  39,  49,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0],
        [ 46,  30,  17,  15,  35,  79,  61,  95,  62,  27,  23,  69,  87,  65,
          68, 100,  73,  33,  22,   9,  81,  56,  63,  97,  57,  77,  75,  72,
          11,   5,   2,  10,  86,  85,  29,  94,  74,  90,  43,  49,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0],
        [  2,  42,  93,  53,  79,  46,  74,  57,  63,  88,  10,  45,   4,  22,
           8,  24,  54,  12,  56,  35,  59,  51,  89,  39,  49,  15,  40,  71,
         101,  83,  67,  34,   9,  23,  82,  72,  50,  21,  95,  98,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0],
        [ 15,  75,  34, 100,  52,  42,  19,  22,  62,  16,   4,  13,  32,  68,
          50,  73,  45,  39,  97,  57,  89,   2

RuntimeError: No active exception to reraise